# IBM Telco Customer Churn: Data Analytics & Feature Pipeline
### Hands-on Practical Pipeline (Stage 1 to Stage 4)
**Target Dataset:** `WA_Fn-UseC_-Telco-Customer-Churn.csv` (7,043 rows, 21 columns)

---
### Notebook Objectives & Structure:
Each stage and individual operation is cleanly separated into its own dedicated cell with explanatory markdown:

- **Stage 1: Problem Definition & Scoping**
  - Business Objective & Loss Mapping (Cost of False Negatives vs False Positives)
  - Success Criteria (Recall $\ge 0.78$, ROC-AUC $\ge 0.84$) & GDPR Compliance
- **Stage 2: Data Ingestion & Low-Level Hygiene**
  - Step 1: File Integrity & Zero-Byte Verification (`Path.stat().st_size`, `hashlib.sha256()`)
  - Step 2: Direct Dataset Ingestion (`pd.read_csv()`)
  - Step 3: Initial Memory Footprint Inspection
  - Step 4: Practical Memory Optimization via Type Downcasting (`int64` $	o$ `int8`/`int16`, `float64` $	o$ `float32`)
  - Step 5: Out-of-Core Chunked Streaming Demonstration
  - Step 6: Export Stage 2 Ingestion CSV (`stage2_ingested_telco.csv`)
- **Stage 3: Data Preprocessing, Cleaning & EDA ("Dirty Data Audit")**
  - Step 1: Schema Standardization (`snake_case` conversion)
  - Step 2: String Normalization & Whitespace Trimming
  - Step 3: Missing Mechanism Analysis (**MCAR, MAR, MNAR**) & whitespace audit in `total_charges`
  - Step 4: Missingness Indicator Flagging (`total_charges_is_missing = 1`)
  - Step 5: Domain & Automated Imputation (`fillna(0.0)`, `SimpleImputer` Median / Mode)
  - Step 6: Deduplication & Target Encoding (`churn`: Yes $	o 1$, No $	o 0$)
  - Step 7: Outlier Detection & Capping (Tukey's IQR Fence Method)
  - Step 8: Exploratory Data Analysis (Summary statistics, correlations, segment churn rates)
  - Step 9: Export Stage 3 Cleaned CSV (`stage3_cleaned_telco.csv`)
- **Stage 4: Feature Engineering, Transformation & Selection**
  - Step 1: PII Removal (`customer_id`) & Strict Stratified Train/Test Split (Fit-Transform Separation)
  - Step 2: Domain Feature Engineering (`charge_per_tenure`, `total_to_monthly_ratio`, `active_services_count`, `has_fiber_no_techsupport`, `contract_loyalty_score`)
  - Step 3: Cardinality Reduction (Grouping rare categories $< 2\%$ to `'Other'`)
  - Step 4: Categorical Encoding (Binary Mapping & `OneHotEncoder(drop='first')`)
  - Step 5: Numerical Feature Scaling (`StandardScaler`)
  - Step 6: Multistage Feature Selection (Zero-Variance, Quasi-Constant, Correlation Filter, Mutual Information `SelectKBest`)
  - Step 7: Dimensionality Reduction via Unsupervised Projection (`PCA(n_components=0.95)`)
  - Step 8: Artifact Serialization (Persisting fitted transformers with `joblib`)
  - Step 9: Export Stage 4 Feature Deliverables (`train_features.csv` & `test_features.csv`)

## 0. Setup & Library Imports
Importing all required standard libraries for file inspection, preprocessing, feature engineering, selection, and serialization.

In [6]:
import re
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Sklearn Imputation, Preprocessing, Decomposition, and Feature Selection
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA

print("Environment setup complete. Ready to execute Stages 1 through 4.")

Environment setup complete. Ready to execute Stages 1 through 4.


---
## Stage 1: Problem Definition & Scoping

### 1. Business Objective Translation
- **Business Problem:** Telecom customer acquisition costs ($300 - $600) drastically exceed retention costs ($20 - $50). Subscriber churn directly damages recurring annual revenue (ARR).
- **ML Formulation:** Supervised Binary Classification (`churn`: 1 = Churned, 0 = Retained).
- **Cost of Errors:**
  - *False Negative (FN):* High cost. A churner is missed and leaves ($500+ lifetime revenue lost).
  - *False Positive (FP):* Low cost. A non-churner receives an unnecessary retention offer ($25 discount cost).
  - *Decision Metric:* Prioritize **Recall $\ge 0.78$** (catch maximum churners) and **ROC-AUC $\ge 0.84$** with well-calibrated probabilities.

### 2. Constraints & Governance
- **GDPR Compliance:** `customerID` is unique pseudonymous personal identifier (PII) and must be dropped prior to modeling.
- **Operational Mode:** Batch scoring monthly prior to customer billing/renewal cycles.

In [7]:
# Stage 1 Scoping verification directly on raw dataset
df_raw_check = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

total_records = len(df_raw_check)
churn_yes = (df_raw_check["Churn"] == "Yes").sum()
churn_no = (df_raw_check["Churn"] == "No").sum()
print(f"Total Cohort Size     : {total_records:,} subscribers")
# print("=" * 60)
# print("STAGE 1 SCOPING & BUSINESS METRICS SUMMARY")
# print("=" * 60)
# print(f"Total Cohort Size     : {total_records:,} subscribers")
# print(f"Retained (Class 0)    : {churn_no:,} ({churn_no/total_records*100:.2f}%)")
# print(f"Churned  (Class 1)    : {churn_yes:,} ({churn_yes/total_records*100:.2f}%)")
# print(f"Class Imbalance Ratio : ~{churn_no/churn_yes:.1f}:1 (Negative : Positive)")
# print(f"Primary Target Metric : Recall >= 0.78 (to minimize lost customer revenue)")
# print(f"Secondary Metric      : ROC-AUC >= 0.84")
# print(f"Regulatory Mandate    : Drop PII 'customerID' under GDPR data minimization")
# print("=" * 60)

Total Cohort Size     : 7,043 subscribers


---
## Stage 2: Data Ingestion & Low-Level Hygiene

**Goal:** Ensure file integrity before touching data, load the dataset cleanly, inspect initial memory usage, aggressively downcast numerical data types (`int64` $	o$ `int8`/`int16`, `float64` $	o$ `float32`) without losing precision to minimize RAM footprint, demonstrate out-of-core chunked streaming, and save the ingested file.

### Step 1: File Integrity & Zero-Byte Verification
- **Zero-Byte Check:** Verifies file size $> 0$ bytes using `pathlib.Path.stat().st_size`.
- **SHA-256 Checksum:** Computes cryptographic hash using `hashlib.sha256()` to ensure payloads are untampered.

In [8]:
# File Integrity & Zero-Byte Verification (Pre-Ingestion Hygiene)
dataset_path = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# # (A) Check physical byte size on disk
# file_size_bytes = dataset_path.stat().st_size
# if file_size_bytes == 0:
#     raise ValueError(f"Zero-Byte Check FAILED: '{dataset_path.name}' is an empty file (0 bytes)!")
# print(f"Zero-Byte Check PASSED: '{dataset_path.name}' ({file_size_bytes:,} bytes, {file_size_bytes / (1024*1024):.2f} MB)")

# (B) Compute cryptographic SHA-256 hash
hasher = hashlib.sha256()
with open(dataset_path, "rb") as f:
    while chunk := f.read(65536):
        hasher.update(chunk)
file_sha256 = hasher.hexdigest()
print(f"SHA-256 Checksum: {file_sha256}")

SHA-256 Checksum: 88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a


### Step 2: Direct Dataset Ingestion
Loading the flat file into memory and verifying initial shape.

In [9]:
# Ingest Raw Dataset into DataFrame
df = pd.read_csv(dataset_path)
# print(f"Dataset Loaded Successfully! Raw Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.shape
df.head(5)




,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Step 3: Initial Memory Footprint Inspection
Checking baseline column types and deep RAM consumption.

In [10]:
# Inspect Column Data Types & Initial Memory Usage
print("Column Data Types Before Downcasting:")
df.info()
print(df.dtypes)

# initial_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
# print(f"\nInitial In-Memory Footprint: {initial_memory_mb:.2f} MB")

Column Data Types Before Downcasting:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  Pape

### Step 4: Practical Memory Optimization (Type Downcasting)
Aggressively downcasting `int64` and `float64` to minimal representations (`int8`, `int16`, `float32`) without precision loss.

In [11]:
# Downcast numeric columns to save memory
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')

for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')
print("\nColumn Data Types After Downcasting:")
print(df.dtypes)
# optimized_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
# savings_pct = (1.0 - (optimized_memory_mb / initial_memory_mb)) * 100

# print(f"Memory AFTER Downcasting : {optimized_memory_mb:.2f} MB")
# print(f"RAM Footprint Reduction  : {savings_pct:.1f}% savings")


Column Data Types After Downcasting:
customerID           object
gender               object
SeniorCitizen          int8
Partner              object
Dependents           object
tenure                 int8
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float32
TotalCharges         object
Churn                object
dtype: object


### Step 5: Out-of-Core Processing Demonstration
Demonstrates streaming large datasets in bounded chunks (`chunksize=2000`) to avoid out-of-memory crashes.

In [12]:
# Demonstrate Out-of-Core Chunked Streaming
chunk_size = 2000
total_streamed = 0
for idx, chunk in enumerate(pd.read_csv(dataset_path, chunksize=chunk_size), start=1):
    total_streamed += len(chunk)
    print(f"Chunk #{idx}: Streamed and processed {len(chunk):,} rows.")

print(f"Total rows verified via out-of-core streaming: {total_streamed:,}")

Chunk #1: Streamed and processed 2,000 rows.
Chunk #2: Streamed and processed 2,000 rows.
Chunk #3: Streamed and processed 2,000 rows.
Chunk #4: Streamed and processed 1,043 rows.
Total rows verified via out-of-core streaming: 7,043


### Step 6: Export Stage 2 Ingestion CSV
Saving the memory-optimized ingested dataset as a pipeline checkpoint.

In [13]:
# Save Stage 2 CSV Deliverable
df.to_csv("stage2_ingested_telco.csv", index=False)
print("[STAGE 2 DELIVERABLE] Saved -> stage2_ingested_telco.csv")

[STAGE 2 DELIVERABLE] Saved -> stage2_ingested_telco.csv


---
## Stage 3: Data Preprocessing, Cleaning & EDA ("Dirty Data Audit")

**Goal:** Clean dirty telemetry: standardize column names to `snake_case`, trim strings, diagnose missing value mechanisms (**MCAR, MAR, MNAR**), flag missingness indicators, apply domain & automated imputation, deduplicate, cap outliers using Tukey's IQR fences, and run exploratory data analysis (EDA).

### Step 1: Schema Standardization (`snake_case`)
Converting all column headers to uniform lowercase `snake_case` using regular expressions.

In [14]:
# Schema Standardization: Convert all column names to snake_case
def to_snake_case(name: str) -> str:
    s = re.sub(r'(?<!^)(?=[A-Z])', '_', name).lower()
    s = s.replace(" ", "_").replace("-", "_")
    return re.sub(r'_+', '_', s)

df.columns = [to_snake_case(col) for col in df.columns]
print("Standardized Column Names (snake_case):")
print(list(df.columns))

Standardized Column Names (snake_case):
['customer_i_d', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_t_v', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn']


### Step 2: String Normalization & Whitespace Trimming
Trimming leading/trailing whitespace across all text/object columns.

In [15]:
# String Normalization: Strip leading & trailing whitespace
for col in df.select_dtypes(include=['object']).columns:
    if col != 'customer_id':
        df[col] = df[col].astype(str).str.strip()

print("String normalization & whitespace trimming complete.")

String normalization & whitespace trimming complete.


### Step 3: Missing Mechanism Analysis (MCAR / MAR / MNAR)
- **MCAR (Missing Completely at Random):** Missingness is pure random noise; unrelated to any variable.
- **MAR (Missing at Random):** Missingness is systematically explained by another observed feature.
  - *Audit:* `total_charges` contains 11 whitespace strings `" "`. All 11 records have `tenure == 0` (brand-new customers who haven't completed a billing cycle) $	o$ **MAR**.
- **MNAR (Missing Not at Random):** Missingness depends directly on the unobserved value itself (e.g. sentinel codes like -999).

In [16]:
# Audit whitespace ' ' strings in total_charges
blank_mask = (df['total_charges'] == ' ') | (df['total_charges'] == '')
print(f"Blank space ' ' count in total_charges: {blank_mask.sum()}")

# Inspect tenure for these blank records
affected_tenures = df.loc[blank_mask, 'tenure'].unique()
print(f"Tenure values for blank total_charges records: {affected_tenures}")
print("""
[DIAGNOSIS]: MAR (Missing At Random)
The missing total_charges occur exclusively when tenure == 0 (brand-new subscribers).
The missingness is fully explained by an observed variable ('tenure').
""")

Blank space ' ' count in total_charges: 11
Tenure values for blank total_charges records: [0]

[DIAGNOSIS]: MAR (Missing At Random)
The missing total_charges occur exclusively when tenure == 0 (brand-new subscribers).
The missingness is fully explained by an observed variable ('tenure').



### Step 4: Missingness Indicator Flagging
Preserving the "signal of absence" prior to filling values so downstream ML models can distinguish brand-new subscribers.

In [17]:
# Missingness Indicator Flagging: Preserve the 'signal of absence'
df['total_charges_is_missing'] = blank_mask.astype(int)
print("Created missingness indicator flag: 'total_charges_is_missing'")
print(f"Flagged records count: {df['total_charges_is_missing'].sum()}")

Created missingness indicator flag: 'total_charges_is_missing'
Flagged records count: 11


### Step 5: Domain & Automated Imputation (Median / Mode / Multivariate)
- **Domain-Specific Imputation (`fillna(0.0)`):** For customers with `tenure == 0`, accumulated charges are logically `0.0`.
- **`SimpleImputer(strategy='median')`:** Automated median imputation for continuous skewed numerics.
- **`SimpleImputer(strategy='most_frequent')`:** Automated mode imputation for categorical columns.
- *Multivariate Patterns:* `KNNImputer()` and `IterativeImputer()` (MICE) documented for multi-feature regression imputation.

In [18]:
# Automated Imputation Pipeline

# (A) Domain-specific Imputation using pandas (fillna):
# For brand-new customers with tenure == 0, accumulated total charges are logically 0.0
df['total_charges'] = pd.to_numeric(df['total_charges'].replace(' ', np.nan))
df['total_charges'] = df['total_charges'].fillna(0.0)
print("Domain imputation: Imputed 0.0 for tenure == 0 records.")

# (B) Automated Median Imputation using SimpleImputer for skewed numerical features
num_cols = df.select_dtypes(include=[np.number]).columns.drop(['total_charges_is_missing'], errors='ignore')
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])
print("SimpleImputer(strategy='median') applied across numerical columns.")

# (C) Automated Mode (Most Frequent) Imputation using SimpleImputer for categoricals
cat_cols = df.select_dtypes(include=['object']).columns
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
    print("SimpleImputer(strategy='most_frequent') applied across categorical columns.")

# (D) Advanced Multivariate Nearest-Neighbor Imputation (KNNImputer) Pattern:
# Uses Euclidean distance across features to predict missing values
# knn_imputer = KNNImputer(n_neighbors=5)
# df[num_cols] = knn_imputer.fit_transform(df[num_cols])

# (E) Advanced Multivariate Chained Equations (IterativeImputer / MICE) Pattern:
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer
# mice_imputer = IterativeImputer(max_iter=10, random_state=42)
# df[num_cols] = mice_imputer.fit_transform(df[num_cols])

print(f"Missing values remaining in dataset: {df.isnull().sum().sum()}")

Domain imputation: Imputed 0.0 for tenure == 0 records.


SimpleImputer(strategy='median') applied across numerical columns.
SimpleImputer(strategy='most_frequent') applied across categorical columns.
Missing values remaining in dataset: 0


### Step 6: Deduplication & Target Mapping
Removing duplicate records and mapping target `churn`: `'Yes'` $	o 1$, `'No'` $	o 0$.

In [19]:
# Deduplication & Target Conversion
initial_len = len(df)
df = df.drop_duplicates()
print(f"Duplicate rows removed: {initial_len - len(df)}")

# Map Target variable Churn to 1 (Yes) and 0 (No)
df['churn'] = df['churn'].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}).astype(int)

churn_dist = df['churn'].value_counts()
print(f"Target 'churn' distribution:")
print(f"  Retained (0): {churn_dist[0]:,} ({churn_dist[0]/len(df)*100:.2f}%)")
print(f"  Churned  (1): {churn_dist[1]:,} ({churn_dist[1]/len(df)*100:.2f}%)")

Duplicate rows removed: 0
Target 'churn' distribution:
  Retained (0): 5,174 (73.46%)
  Churned  (1): 1,869 (26.54%)


### Step 7: Outlier Detection & Capping (Tukey's IQR Fence Method)
Calculating Tukey's fences $[Q_1 - 1.5 	imes IQR, \ Q_3 + 1.5 	imes IQR]$ to cap extreme numerical values without dropping data.

In [20]:
# Outlier Detection & Capping (Tukey's IQR Fences)
continuous_cols = ['tenure', 'monthly_charges', 'total_charges']

for col in continuous_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    print(f"{col:15s} -> Outliers detected: {outliers_count} (Clipping to [{lower_bound:.2f}, {upper_bound:.2f}])")
    
    # Cap values at fences
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

tenure          -> Outliers detected: 0 (Clipping to [-60.00, 124.00])
monthly_charges -> Outliers detected: 0 (Clipping to [-46.02, 171.37])
total_charges   -> Outliers detected: 0 (Clipping to [-4683.52, 8868.67])


### Step 8: Exploratory Data Analysis (EDA)
Inspecting continuous feature distributions, correlation with `churn`, and churn rates across service segments.

In [21]:
# Exploratory Data Analysis (EDA)
print("=== Continuous Features Summary Statistics ===")
print(df[continuous_cols].describe().round(2))

print("\n=== Correlation with Target 'churn' ===")
num_corr_cols = continuous_cols + ['senior_citizen', 'total_charges_is_missing', 'churn']
print(df[num_corr_cols].corr()['churn'].sort_values(ascending=False).round(4))

print("\n=== Churn Rate by Contract Type ===")
print(df.groupby('contract')['churn'].agg(Total='count', Churned='sum', Rate=lambda x: f"{x.mean()*100:.2f}%"))

print("\n=== Churn Rate by Internet Service ===")
print(df.groupby('internet_service')['churn'].agg(Total='count', Churned='sum', Rate=lambda x: f"{x.mean()*100:.2f}%"))

=== Continuous Features Summary Statistics ===
        tenure  monthly_charges  total_charges
count  7043.00          7043.00        7043.00
mean     32.37            64.76        2279.73
std      24.56            30.09        2266.79
min       0.00            18.25           0.00
25%       9.00            35.50         398.55
50%      29.00            70.35        1394.55
75%      55.00            89.85        3786.60
max      72.00           118.75        8684.80

=== Correlation with Target 'churn' ===
churn                       1.0000
monthly_charges             0.1934
senior_citizen              0.1509
total_charges_is_missing   -0.0238
total_charges              -0.1983
tenure                     -0.3522
Name: churn, dtype: float64

=== Churn Rate by Contract Type ===
                Total  Churned    Rate
contract                              
Month-to-month   3875     1655  42.71%
One year         1473      166  11.27%
Two year         1695       48   2.83%

=== Churn Rate by 

### Step 9: Export Stage 3 Cleaned CSV
Saving the sanitized dataset as the baseline input for Stage 4.

In [22]:
# Save Stage 3 CSV Deliverable
df.to_csv("stage3_cleaned_telco.csv", index=False)
print("[STAGE 3 DELIVERABLE] Saved -> stage3_cleaned_telco.csv")

[STAGE 3 DELIVERABLE] Saved -> stage3_cleaned_telco.csv


---
## Stage 4: Feature Engineering, Transformation & Selection

**Goal:** Generate domain features, manage cardinality, encode categoricals, scale numerics, select top features, apply PCA dimensionality reduction, and serialize fitted transformer artifacts.

### Step 1: PII Removal & Stratified Train/Test Split
Dropping `customer_id` (PII) and performing an 80/20 stratified split *first* to enforce strict fit-transform separation and prevent data leakage.

In [23]:
# 1. Drop PII identifier robustly (case-insensitive check)
id_cols = [c for c in df.columns if c.lower() in ['customerid', 'customer_id', 'id']]
if id_cols:
    df = df.drop(columns=id_cols)
    print(f"Successfully dropped PII columns: {id_cols}")

target_col = 'churn' if 'churn' in df.columns else 'Churn'
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. Stratified Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Data Leakage Audit: Check overlap on the ORIGINAL indices BEFORE resetting!
original_overlap = set(X_train.index) & set(X_test.index)
assert len(original_overlap) == 0, f"Data leakage detected! {len(original_overlap)} overlapping rows!"
print(f"Data Leakage Audit Passed: 0 overlapping indices. Train: {len(X_train):,}, Test: {len(X_test):,}")

# 4. Now safely reset indices
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

Data Leakage Audit Passed: 0 overlapping indices. Train: 5,634, Test: 1,409


### Step 2: Domain Feature Engineering
Creating high-signal telecom domain features:
- `charge_per_tenure`: monthly spend rate relative to tenure (safe division for `tenure = 0`).
- `total_to_monthly_ratio`: total charges divided by monthly charges (estimated billing cycles).
- `active_services_count`: sum of active digital add-ons.
- `has_fiber_no_techsupport`: interaction flag identifying the highest-risk churn profile.
- `contract_loyalty_score`: ordinal ranking (`Month-to-month` = 1, `One year` = 2, `Two year` = 3).

In [24]:

# Domain Feature Generation
def create_domain_features(data: pd.DataFrame) -> pd.DataFrame:
    df_out = data.copy()

    # Fix column naming mismatch from earlier snake_case conversion
    if 'streaming_t_v' in df_out.columns and 'streaming_tv' not in df_out.columns:
        df_out.rename(columns={'streaming_t_v': 'streaming_tv'}, inplace=True)

    # Ratio: monthly spend rate relative to tenure (safe division for tenure=0)
    safe_t = df_out['tenure'].replace(0, 1)
    df_out['charge_per_tenure'] = df_out['monthly_charges'] / safe_t

    # Ratio: Total to monthly charges (approximates actual bill cycles)
    safe_m = df_out['monthly_charges'].replace(0, 1.0)
    df_out['total_to_monthly_ratio'] = df_out['total_charges'] / safe_m

    # Aggregation: count of active digital add-on services
    services = [
        'online_security',
        'online_backup',
        'device_protection',
        'tech_support',
        'streaming_tv',
        'streaming_movies'
    ]

    df_out['active_services_count'] = (
        df_out[services] == 'Yes'
    ).sum(axis=1)

    # Risk Interaction: Fiber optic user with no tech support
    df_out['has_fiber_no_techsupport'] = (
        (df_out['internet_service'] == 'Fiber optic') &
        (df_out['tech_support'] == 'No')
    ).astype(int)

    # Contract Loyalty Score (Ordinal ranking)
    contract_map = {
        'Month-to-month': 1,
        'One year': 2,
        'Two year': 3
    }

    df_out['contract_loyalty_score'] = (
        df_out['contract'].map(contract_map).fillna(1).astype(int)
    )

    return df_out


# Generate domain features
X_train = create_domain_features(X_train)
X_test = create_domain_features(X_test)

print(f"Domain features created. Total feature columns: {X_train.shape[1]}")


Domain features created. Total feature columns: 26


### Step 3: Cardinality Reduction (Rare Category Grouping)
Grouping infrequent categorical levels (frequency $< 2\%$) into `'Other'` to prevent dimension explosion.

In [25]:
# Cardinality Reduction: Group rare categories (< 2%) into 'Other'
rare_threshold = 0.02
cat_cols_to_check = ['payment_method', 'contract', 'internet_service']

for col in cat_cols_to_check:
    freq = X_train[col].value_counts(normalize=True)
    frequent_categories = freq[freq >= rare_threshold].index.tolist()
    
    # Apply to train fold
    X_train[col] = X_train[col].apply(lambda x: x if x in frequent_categories else 'Other')
    # Apply fitted frequent categories to test fold
    X_test[col] = X_test[col].apply(lambda x: x if x in frequent_categories else 'Other')

print("Rare category grouping complete.")

Rare category grouping complete.


### Step 4: Categorical Encoding (Binary Mapping & One-Hot Encoding)
- Binary mapping for 2-level categories (`partner`, `dependents`, `phone_service`, `paperless_billing`, `gender`).
- Low-cardinality `OneHotEncoder(drop='first', sparse_output=False)` fit strictly on the training fold.

In [26]:
# Categorical Encoding Pipeline

# (A) Binary Mapping for 2-level categories
binary_cols = ['partner', 'dependents', 'phone_service', 'paperless_billing']
for col in binary_cols:
    mapping = {'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}
    X_train[col] = X_train[col].map(mapping).fillna(0).astype(int)
    X_test[col] = X_test[col].map(mapping).fillna(0).astype(int)

# Map gender
X_train['gender'] = X_train['gender'].map({'Male': 1, 'Female': 0, 'male': 1, 'female': 0}).fillna(0).astype(int)
X_test['gender'] = X_test['gender'].map({'Male': 1, 'Female': 0, 'male': 1, 'female': 0}).fillna(0).astype(int)

# (B) One-Hot Encoding for multi-class nominal variables
nominal_cols = [
    'multiple_lines', 'internet_service', 'online_security', 'online_backup',
    'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
    'contract', 'payment_method'
]

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[nominal_cols])

def apply_ohe(df_data: pd.DataFrame, encoder: OneHotEncoder, cols: list) -> pd.DataFrame:
    encoded_arr = encoder.transform(df_data[cols])
    encoded_cols = encoder.get_feature_names_out(cols)
    encoded_df = pd.DataFrame(encoded_arr, columns=encoded_cols, index=df_data.index)
    return pd.concat([df_data.drop(columns=cols), encoded_df], axis=1)

X_train = apply_ohe(X_train, ohe, nominal_cols)
X_test = apply_ohe(X_test, ohe, nominal_cols)

# Align columns between train and test
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0.0)
print(f"Features after Categorical Encoding: {X_train.shape[1]}")

Features after Categorical Encoding: 37


### Step 5: Numerical Feature Scaling (`StandardScaler`)
Standardizing continuous variables to mean 0 and unit variance $\sigma = 1$.

In [27]:
# Numerical Feature Scaling using StandardScaler
scale_cols = ['tenure', 'monthly_charges', 'total_charges', 'charge_per_tenure', 'total_to_monthly_ratio']
scaler = StandardScaler()
scaler.fit(X_train[scale_cols])

X_train[scale_cols] = scaler.transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])
print("StandardScaler fitted on train fold and applied to train & test.")

StandardScaler fitted on train fold and applied to train & test.


### Step 6: Multistage Feature Selection
- **Zero-Variance Filter:** Eliminates columns with variance $= 0$.
- **Quasi-Constant Filter:** Drops columns where over $99\%$ of values are identical.
- **Collinearity Filter:** Drops one of any feature pair with Pearson correlation $|r| > 0.85$.
- **Mutual Information `SelectKBest`:** Isolates the top 15 most informative non-linear predictors.

In [28]:
# ==============================================================================
# Step 6: Multistage Feature Selection Pipeline
# ==============================================================================
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
import numpy as np

# Defensive check: Ensure X_train and X_test contain strictly numeric dtypes
non_numeric_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    print(f"[Warning] Dropping non-numeric residual columns before feature selection: {non_numeric_cols}")
    X_train = X_train.drop(columns=non_numeric_cols)
    X_test = X_test.drop(columns=non_numeric_cols)

# (A) Zero-Variance Filter
vt_zero = VarianceThreshold(threshold=0.0)
vt_zero.fit(X_train)
X_train = X_train.loc[:, vt_zero.get_support()]
X_test = X_test.loc[:, vt_zero.get_support()]

# (B) Quasi-Constant Filter (threshold=0.01)
vt_quasi = VarianceThreshold(threshold=0.01)
vt_quasi.fit(X_train)
X_train = X_train.loc[:, vt_quasi.get_support()]
X_test = X_test.loc[:, vt_quasi.get_support()]

# (C) Collinearity Filter: Drop one of any feature pair with correlation r > 0.85
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [c for c in upper.columns if any(upper[c] > 0.85)]
if high_corr_cols:
    X_train = X_train.drop(columns=high_corr_cols)
    X_test = X_test.drop(columns=high_corr_cols)
    print(f"Dropped {len(high_corr_cols)} collinear features (r > 0.85): {high_corr_cols}")

# (D) Statistical Selection via Mutual Information (SelectKBest, top k=15)
TOP_K = min(15, X_train.shape[1])
selector = SelectKBest(score_func=mutual_info_classif, k=TOP_K)
selector.fit(X_train, y_train)

selected_features = X_train.columns[selector.get_support()].tolist()
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"\nTop {TOP_K} Features Selected via Mutual Information:")
for idx, col in enumerate(selected_features, 1):
    print(f"  {idx:2d}. {col}")

[Warning] Dropping non-numeric residual columns before feature selection: ['customer_i_d']
Dropped 9 collinear features (r > 0.85): ['total_to_monthly_ratio', 'multiple_lines_No phone service', 'online_security_No internet service', 'online_backup_No internet service', 'device_protection_No internet service', 'tech_support_No internet service', 'streaming_tv_No internet service', 'streaming_movies_No internet service', 'contract_Two year']

Top 15 Features Selected via Mutual Information:
   1. dependents
   2. tenure
   3. paperless_billing
   4. monthly_charges
   5. total_charges
   6. charge_per_tenure
   7. active_services_count
   8. has_fiber_no_techsupport
   9. contract_loyalty_score
  10. internet_service_Fiber optic
  11. internet_service_No
  12. online_security_Yes
  13. tech_support_Yes
  14. contract_One year
  15. payment_method_Electronic check


### Step 7: Dimensionality Reduction via Unsupervised Projection (`PCA`)
Compressing high-dimensional correlated feature matrices into orthogonal components while preserving **95% cumulative explained variance** using `sklearn.decomposition.PCA`.

In [29]:
# Dimensionality Reduction (Unsupervised Projection via PCA)
# Compressing high-dimensional correlated feature matrices into orthogonal components
# Preserving 95% cumulative explained variance
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_selected)
X_test_pca = pca.transform(X_test_selected)

explained_var = pca.explained_variance_ratio_
print(f"[PCA PROJECTION] Compressed {X_train_selected.shape[1]} features -> {X_train_pca.shape[1]} orthogonal components.")
print(f"Explained Variance per Component: {[round(v, 4) for v in explained_var]}")
print(f"Total Cumulative Explained Variance: {np.sum(explained_var)*100:.2f}%")

[PCA PROJECTION] Compressed 15 features -> 9 orthogonal components.
Explained Variance per Component: [0.5441, 0.185, 0.0766, 0.0589, 0.0217, 0.0201, 0.0185, 0.0171, 0.0152]
Total Cumulative Explained Variance: 95.73%


### Step 8: Artifact Serialization (`joblib`)
Persisting fitted transformer objects (`ohe`, `scaler`, `selector`, `pca`, `selected_features`) to disk using `joblib.dump()` so production inference pipelines apply identical parameters without retraining.

In [30]:
# Artifact Serialization: Persisting fitted transformers with joblib
artifacts_dir = Path("fitted_artifacts")
artifacts_dir.mkdir(exist_ok=True)

fitted_transformers = {
    "ohe_encoder.joblib": ohe,
    "scaler.joblib": scaler,
    "mi_selector.joblib": selector,
    "pca.joblib": pca,
    "selected_features.joblib": selected_features
}

for filename, transformer_obj in fitted_transformers.items():
    joblib.dump(transformer_obj, artifacts_dir / filename)

print(f"[ARTIFACT SERIALIZATION] Successfully saved {len(fitted_transformers)} fitted transformers to '{artifacts_dir.name}/'.")

[ARTIFACT SERIALIZATION] Successfully saved 5 fitted transformers to 'fitted_artifacts/'.


### Step 9: Export Stage 4 Feature Deliverables
Saving the final engineered train and test matrices (`train_features.csv` & `test_features.csv`).

In [31]:
# Save Stage 4 CSV Deliverables
train_export = pd.concat([X_train_selected, y_train.rename('churn')], axis=1)
test_export = pd.concat([X_test_selected, y_test.rename('churn')], axis=1)

train_export.to_csv("train_features.csv", index=False)
test_export.to_csv("test_features.csv", index=False)

print("[STAGE 4 DELIVERABLES] Saved -> train_features.csv & test_features.csv")
print("""
================================================================================
STAGES 1 THROUGH 4 COMPLETED SUCCESSFULLY!
Final Train Feature Matrix: {} rows x {} columns
Final Test Feature Matrix : {} rows x {} columns
PCA-Compressed Features   : {} orthogonal components (95% variance)
Features & Transformers Ready for Production Modeling Phase!
================================================================================
""".format(X_train_selected.shape[0], X_train_selected.shape[1], X_test_selected.shape[0], X_test_selected.shape[1], X_train_pca.shape[1]))

[STAGE 4 DELIVERABLES] Saved -> train_features.csv & test_features.csv

STAGES 1 THROUGH 4 COMPLETED SUCCESSFULLY!
Final Train Feature Matrix: 5634 rows x 15 columns
Final Test Feature Matrix : 1409 rows x 15 columns
PCA-Compressed Features   : 9 orthogonal components (95% variance)
Features & Transformers Ready for Production Modeling Phase!

